# Scratch – repr & visualisation testing

Temporary notebook for testing `_repr_html_` / `__repr__` across all classes and visualisation helpers like `coverage_bar()` and `tree()`.

**Delete when done.**

In [ ]:
from datetime import datetime, timezone

import numpy as np

from timedatamodel import (
    AggregationMethod,
    CoverageBar,
    DataType,
    Dimension,
    Frequency,
    GeoLocation,
    HierarchicalTimeSeries,
    HierarchyNode,
    TimeSeries,
    TimeSeriesArray,
    TimeSeriesCollection,
    TimeSeriesTable,
    TimeSeriesType,
)

## Helper – generate hourly timestamps

In [ ]:
def hourly_ts(n=24, start="2024-01-15"):
    base = datetime.fromisoformat(f"{start}T00:00:00+00:00")
    from datetime import timedelta
    return [base + timedelta(hours=i) for i in range(n)]

---
## 1. TimeSeries

In [ ]:
ts = TimeSeries(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=hourly_ts(24),
    values=[120 + 5 * np.sin(i) for i in range(24)],
    name="power",
    unit="MW",
    data_type=DataType.MEASUREMENT,
    description="Hourly power output from wind farm Alpha",
)
ts

In [ ]:
print(ts)

### TimeSeries – empty

In [ ]:
ts_empty = TimeSeries(Frequency.PT1H, timestamps=[], values=[], name="empty")
ts_empty

### TimeSeries – with NaN gaps

In [ ]:
vals_gap = [float(i) for i in range(24)]
for i in [5, 6, 7, 15, 16]:
    vals_gap[i] = None

ts_gap = TimeSeries(
    Frequency.PT1H,
    timestamps=hourly_ts(24),
    values=vals_gap,
    name="gappy_signal",
    unit="kW",
)
ts_gap

### TimeSeries – short (no truncation)

In [ ]:
ts_short = TimeSeries(
    Frequency.PT1H,
    timestamps=hourly_ts(5),
    values=[1.0, 2.0, 3.0, 4.0, 5.0],
    name="short",
)
ts_short

### TimeSeries – with location and labels

In [ ]:
ts_loc = TimeSeries(
    Frequency.PT1H,
    timestamps=hourly_ts(24),
    values=[100 + i * 0.5 for i in range(24)],
    name="temperature",
    unit="°C",
    location=GeoLocation(latitude=59.91, longitude=10.75),
    labels={"source": "met.no", "station": "Oslo-Blindern"},
    data_type=DataType.MEASUREMENT,
    description="Air temperature at Blindern station",
)
ts_loc

---
## 2. TimeSeriesTable

In [ ]:
timestamps = hourly_ts(24)
tbl = TimeSeriesTable(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=timestamps,
    values=np.column_stack([
        [120 + 5 * np.sin(i) for i in range(24)],
        [80 + 3 * np.cos(i) for i in range(24)],
        [200 + 10 * np.sin(i + 1) for i in range(24)],
    ]),
    names=["wind_farm_A", "wind_farm_B", "solar_park_C"],
    units=["MW", "MW", "MW"],
    data_types=[DataType.MEASUREMENT, DataType.MEASUREMENT, DataType.FORECAST],
)
tbl

In [ ]:
print(tbl)

---
## 3. TimeSeriesCollection

In [ ]:
ts_a = TimeSeries(
    Frequency.PT1H, timestamps=hourly_ts(24), values=[float(i) for i in range(24)],
    name="series_A", unit="MW",
)
ts_b = TimeSeries(
    Frequency.P1D,
    timestamps=[datetime.fromisoformat(f"2024-01-{d:02d}T00:00:00+00:00") for d in range(1, 8)],
    values=[100.0, 105.0, 98.0, None, 110.0, 115.0, 108.0],
    name="series_B", unit="GWh",
)

coll = TimeSeriesCollection(
    [ts_a, ts_b, tbl],
    name="My Collection",
    description="A mix of different series",
)
coll

In [ ]:
print(coll)

---
## 4. TimeSeriesArray (N-dimensional)

In [ ]:
time_labels = hourly_ts(12)
scenarios = ["low", "mid", "high"]

arr = TimeSeriesArray(
    Frequency.PT1H,
    name="price_scenarios",
    unit="EUR/MWh",
    data_type=DataType.SCENARIO,
    dimensions=[
        Dimension(name="timestamp", labels=time_labels),
        Dimension(name="scenario", labels=scenarios),
    ],
    values=np.random.default_rng(42).uniform(30, 80, size=(12, 3)),
)
arr

In [ ]:
print(arr)

### TimeSeriesArray – 1D

In [ ]:
arr_1d = TimeSeriesArray(
    Frequency.PT1H,
    name="load",
    unit="MW",
    dimensions=[Dimension(name="timestamp", labels=hourly_ts(8))],
    values=np.array([100, 105, 110, 108, 103, 99, 95, 92], dtype=float),
)
arr_1d

---
## 5. CoverageBar

In [ ]:
ts_gap.coverage_bar()

In [ ]:
print(ts_gap.coverage_bar())

In [ ]:
tbl.coverage_bar()

In [ ]:
print(tbl.coverage_bar())

---
## 6. HierarchicalTimeSeries & tree()

In [ ]:
timestamps = hourly_ts(24)
rng = np.random.default_rng(0)

def make_ts(name):
    return TimeSeries(
        Frequency.PT1H, timestamps=timestamps,
        values=rng.uniform(50, 200, 24).tolist(),
        name=name, unit="MW", data_type=DataType.MEASUREMENT,
    )

tree_dict = {
    "total": {
        "Norway": {"Bergen": "bergen", "Oslo": "oslo", "Trondheim": "trondheim"},
        "Sweden": {"Stockholm": "stockholm", "Gothenburg": "gothenburg"},
    }
}
series_map = {
    "bergen": make_ts("Bergen"),
    "oslo": make_ts("Oslo"),
    "trondheim": make_ts("Trondheim"),
    "stockholm": make_ts("Stockholm"),
    "gothenburg": make_ts("Gothenburg"),
}

hts = HierarchicalTimeSeries.from_dict(
    tree_dict, series_map,
    levels=["region", "country", "city"],
    name="Nordic Wind Power",
    description="Hierarchical wind power across Nordic cities",
    aggregation=AggregationMethod.SUM,
)
hts

In [ ]:
print(hts)

### HierarchyTree – HTML

In [ ]:
hts.tree()

In [ ]:
print(hts.tree())

---
## 7. Side-by-side: light vs dark

Quick way to preview both colour schemes without changing your OS setting — wrap the HTML repr in a `<div>` that forces a colour scheme.

In [ ]:
from IPython.display import HTML

light = f'<div style="color-scheme: light; padding: 10px;">{ts._repr_html_()}</div>'
dark = f'<div style="color-scheme: dark; background: #0f172a; padding: 10px; border-radius: 8px;">{ts._repr_html_()}</div>'

HTML(f"<h4>Light mode</h4>{light}<br><h4>Dark mode</h4>{dark}")